### Profiling

> Inputs : 

For historical: 
* processed/user_profile_candidates.parquet 
* processed/reviews_clean.parquet 
* processed/recipe_lookup.parquet
* processed/data/baselines.json

For synthetic: 
* processed/catalog.parquet 

> Outputs: 

* processed/profiles_historical.jsonl
* processed/profiles_synthetic.jsonl
* processed/profiles_historical_summary.csv
* processed/profiles_synthetic_summary.csv

In [1]:
import json
import re
from collections import Counter

import numpy as np
import pandas as pd

from config import (PROC_DIR, SEED, GOAL_RULES, LIKED_RATING, DISLIKED_MAX_RATING, read_table, parse_list_cols, 
                    render_profile_text, write_jsonl, split_items, snippet,constraint_mask, verified_mask, goal_mask, term_mask)

### HISTORICAL profiles (30) from HUMMUS review history.

Principle: a profile claim is only made when the evidence clearly differs from what a typical reviewed recipe looks
like (z-test vs review-weighted baseline). Everything inferred is tagged with status + confidence; health goals stay
'unknown' unless clearly supported. Nothing is treated as confirmed.

* Inputs : processed/user_profile_candidates, reviews_clean, recipe_lookup, baselines.json

* Outputs: processed/profiles_historical.jsonl, profiles_historical_summary.csv

In [2]:
#  config
N_HISTORICAL = 30                      # total profiles, spread evenly over 2 richness x 2 signal strata
SPARSE_RANGE, RICH_RANGE = (20, 50), (100, 300)
MIN_RATING_STD, MIN_LIKED, MIN_SPAN_YEARS, MIN_TEXT = 0.5, 10, 1.0, 15
MIN_N_TENDENCY, Z_TENDENCY, Z_STRONG = 10, 3.0, 5.0
MIN_ITEM_COUNT, Z_ITEM, MIN_LIFT, TOP_TAGS, TOP_INGS = 4, 3.0, 1.8, 6, 8
MIN_DISLIKED = 10

#### Patching for historical profiles: Keep ONLY real taste evidence in "likes" / "dislikes".

Besides parsing artefacts, the likes still contained
  * diet tags ("vegan", "vegetarian", ...)      -> diets are handled as constraints, never as taste
  * health / nutrition tags ("high-in-something", "low-fat", "diabetic", ...) -> handled by nutrition tendencies
  * equipment / process / occasion tags ("bread-machine", "microwave", "from-scratch", "seasonal", "fall", ...)
  * over-narrow dish-type tags ("punch")

The following code uses an explicit blocklist PLUS regex patterns, applied to likes AND dislikes. 
It does NOT change which users are selected, only the likes/dislikes lists and the profile text.

In [3]:
# Items that must never be reported as taste evidence (parsing artefacts, process/equipment/occasion tags, health tags).
JUNK_ITEMS = {
    # parsing artefacts / generic pantry
    "use more or less", "ice cubes", "ice", "water", "warm water", "cold water", "ice water", "hot water", "boiling water",
    "salt", "pepper", "salt and pepper", "meat", "ingredients", "optional", "garnish",
    # diets (handled as constraints) and health tags (handled by nutrition tendencies)
    "vegan", "vegetarian", "gluten-free", "dairy-free", "nut-free", "egg-free", "lactose-free", "kosher", "halal",
    "diabetic", "healthy", "high-in-something", "high-in-something-diabetic-friendly", "low-in-something", "heart-healthy",
    "low-saturated-fat", "very-low-carbs", "healthy", "high-calcium", "low-carb", "low-sodium", "low-fat",
    "low-calorie", "low-cholesterol", "high-protein", "high-fiber", "low-protein",
    # equipment / process / convenience
    "equipment", "bread-machine", "microwave", "mixer", "blender", "food-processor-blender", "crock-pot-slow-cooker",
    "slow-cooker", "dutch-oven", "pressure-cooker", "stove-top", "oven", "grilling", "from-scratch", "oamc-freezer-make-ahead",
    "freezer", "make-ahead", "easy", "to-go", "inexpensive", "one-dish-meal", "for-large-groups", "for-1-or-2", "small-appliance",
    # occasion / season / mood
    "from-scratch", "romantic", "seasonal", "gifts", "holiday-event", "fall", "spring", "summer", "winter", "christmas", "thanksgiving",
    "valentines-day", "st-patricks-day", "new-years", "4th-of-july", "superbowl", "taste-mood", "comfort-food",
    # over-narrow dish-type tags
    "punch", "mixer-drink",
}
JUNK_PATTERNS = re.compile(
    r"^(?:low|very-low|high|reduced|no)[- ](?:in-|carb|fat|sodium|calorie|cholesterol|sugar|protein|fiber|saturated)|"
    r"-free$|-friendly$|something|^\d+-|-or-less$|^(?:number|time)-", re.I)


def is_junk_item(item: str) -> bool:
    return len(item) < 3 or item in JUNK_ITEMS or bool(JUNK_PATTERNS.search(item))

PATTERN_MIN_N, PATTERN_MIN_SHARE, PATTERN_Z = 15, 0.95, 3.0

In [4]:
PCT_METRICS = {  # label -> (lookup column, human meaning of 'higher')
    "calories": "calories_pct", "sodium": "sodium_pct", "sugar": "sugars_pct", "saturated fat": "satfat_pct",
    "protein": "protein_pct", "fiber": "fiber_pct", "cooking time": "duration_pct", "overall healthiness score": "who_pct",
}
RULES = {"vegetarian": "rule_vegetarian", "vegan": "rule_vegan", "dairy-free": "rule_dairy_free",
         "gluten-free": "rule_gluten_free", "nut-free": "rule_nut_free"}
IMPLIES = {"vegan": {"vegetarian", "dairy-free"}}

MENTION_LABELS = {
    "vegetarian": r"\bvegetarian\b", "vegan": r"\bvegan\b", "gluten-free": r"\b(?:gluten|celiac|coeliac)\b",
    "dairy-free": r"\b(?:lactose|dairy[- ]free)\b", "nut-free": r"\bnut[- ]free\b|\bnut allerg\w*",
    "low-carb/keto": r"\b(?:keto|low[- ]carb)\b", "kosher/halal": r"\b(?:kosher|halal)\b", "allergy (unspecified)": r"\ballerg\w*",
}
HEALTH_RE = re.compile(r"\b(?:diabet\w*|blood sugar|cholesterol|low[- ]fat|low[- ]sodium|low[- ]calorie|weight loss|healthy|healthier)\b", re.I)
HOUSEHOLD_RE = re.compile(r"\b(?:my kids?|my children|my son|my daughter|my husband|my wife|my family|toddler\w*|picky|my boyfriend|my girlfriend)\b", re.I)


In [5]:
def allocate(total, sizes):
    """Round-robin allocation so strata stay balanced (e.g. 30 over 4 strata -> 8,8,7,7), capped by availability."""
    alloc = {k: 0 for k in sizes}
    while sum(alloc.values()) < total:
        moved = False
        for k in alloc:
            if sum(alloc.values()) >= total:
                break
            if alloc[k] < sizes[k]:
                alloc[k] += 1
                moved = True
        if not moved:
            break
    return alloc


def select_users(us: pd.DataFrame) -> pd.DataFrame:
    e = us[(us["rating_std"] >= MIN_RATING_STD) & (us["n_liked"] >= MIN_LIKED) & (us["span_years"] >= MIN_SPAN_YEARS)
           & (us["n_text"] >= MIN_TEXT)].copy()
    e["richness"] = np.select([e["n_reviews"].between(*SPARSE_RANGE), e["n_reviews"].between(*RICH_RANGE)], ["sparse", "rich"], "other")
    e["signal"] = np.select([(e["n_restriction"] >= 2) | (e["n_health"] >= 3), (e["n_restriction"] == 0) & (e["n_health"] <= 1)],
                            ["with_signal", "no_signal"], "other")
    e = e[(e["richness"] != "other") & (e["signal"] != "other")]
    print("eligible users per stratum:\n", e.groupby(["richness", "signal"]).size().to_string())
    sizes = e.groupby(["richness", "signal"]).size().to_dict()
    alloc = allocate(N_HISTORICAL, sizes)
    print("allocation per stratum:", alloc)
    sel = [g.sample(alloc[k], random_state=SEED) for k, g in e.groupby(["richness", "signal"]) if alloc[k] > 0]
    if sum(alloc.values()) < N_HISTORICAL:
        print(f"WARNING: only {sum(alloc.values())} eligible users available - relax the thresholds at the top of the file")
    return pd.concat(sel).reset_index(drop=True)


def item_lifts(lists, base_rates, min_count, top_k):
    lists = [l for l in lists if l]
    n = len(lists)
    if n < MIN_N_TENDENCY:
        return [], n
    cnt = Counter(i for l in lists for i in set(l))
    default = min(base_rates.values()) if base_rates else 0.001
    out = []
    for item, c in cnt.items():
        if c < min_count or is_junk_item(item):
            continue
        r, b = c / n, min(max(base_rates.get(item, default), 1e-4), 0.999)
        z = (r - b) / np.sqrt(b * (1 - b) / n)
        if z >= Z_ITEM and r / b >= MIN_LIFT:
            out.append({"item": item, "count": int(c), "n_recipes": n, "rate": round(r, 3), "baseline_rate": round(b, 4),
                        "lift": round(r / b, 2), "z": round(float(z), 1), "status": "observed_pattern"})
    out.sort(key=lambda d: -d["z"])
    return out[:top_k], n

def item_contrasts(liked_lists, disliked_lists, min_count=4, top_k=4):
    """Find items more prevalent in a user's disliked recipes than liked recipes."""
    liked_lists = [set(x) for x in liked_lists if x]
    disliked_lists = [set(x) for x in disliked_lists if x]

    n_liked = len(liked_lists)
    n_disliked = len(disliked_lists)

    if min(n_liked, n_disliked) < MIN_N_TENDENCY:
        return [], (n_liked, n_disliked)

    liked_count = Counter(
        item for items in liked_lists for item in items
    )
    disliked_count = Counter(
        item for items in disliked_lists for item in items
    )

    out = []

    for item, d_count in disliked_count.items():
        if d_count < min_count or is_junk_item(item):
            continue

        p_disliked = d_count / n_disliked
        p_liked = liked_count.get(item, 0) / n_liked

        # Two-proportion z-score using the pooled rate.
        l_count = liked_count.get(item, 0)
        pooled = (d_count + l_count) / (n_disliked + n_liked)

        se = np.sqrt(max(
            pooled * (1 - pooled) *
            (1 / n_disliked + 1 / n_liked),
            1e-12
        ))
        z = (p_disliked - p_liked) / se

        # Avoid division by zero for items absent from liked recipes.
        lift = p_disliked / max(p_liked, 1 / n_liked)

        if (
            p_disliked > p_liked
            and z >= Z_ITEM
            and lift >= MIN_LIFT
        ):
            out.append({
                "item": item,
                "count": int(d_count),
                "n_recipes": n_disliked,
                "rate": round(p_disliked, 3),
                "baseline_rate": round(p_liked, 4),
                "lift": round(lift, 2),
                "z": round(float(z), 1),
                "status": "observed_pattern",
            })

    out.sort(key=lambda x: -x["z"])
    return out[:top_k], (n_liked, n_disliked)

In [ ]:
def build_profile(i, uid, label, rows, lookup, base, ustats):
    r = rows.merge(lookup, on="new_recipe_id", how="left")
    liked = r[r["liked"]]
    disliked = r[(r["rating"] <= DISLIKED_MAX_RATING) & (r["rating_type"] == "stars")]
    coverage = float(r["duration"].notna().mean())
    ev, unknown = [], ["confirmed_dietary_restrictions", "allergies", "budget"]

    # --- taste
    likes_tags, n_t = item_lifts([split_items(s) for s in liked["tags_str"]], base["tag_rates"], MIN_ITEM_COUNT, TOP_TAGS)
    likes_ing, n_i = item_lifts([split_items(s) for s in liked["ing_str"]], base["ingredient_rates"], MIN_ITEM_COUNT, TOP_INGS)
    for x in likes_tags:
        x["kind"] = "tag"
    for x in likes_ing:
        x["kind"] = "ingredient"
    likes = sorted(likes_tags + likes_ing, key=lambda d: (-d["z"], -d["count"], d["item"]),)
    dislikes = []
    if len(disliked) >= MIN_DISLIKED:
        d_tags, _ = item_contrasts([split_items(s) for s in liked["tags_str"]], [split_items(s) for s in disliked["tags_str"]], min_count=4, top_k=3)
        d_ing, _ = item_contrasts([split_items(s) for s in liked["ing_str"]], [split_items(s) for s in disliked["ing_str"]], min_count=4, top_k=4)
        for x in d_tags:
            x["kind"] = "tag"
        for x in d_ing:
            x["kind"] = "ingredient"
        dislikes = d_tags + d_ing

        liked_items = {x["item"] for x in likes}

        dislikes = [
            x for x in dislikes
            if x["item"] not in liked_items
        ]

        # After filtering and removing overlaps:
        if not dislikes:
            unknown.append(
                "dislikes (no reliable pattern met the evidence thresholds)"
            )
    else:
        unknown.append(f"dislikes (only {len(disliked)} low-rated recipes); {MIN_DISLIKED} required)")

    # --- nutrition tendencies (liked recipes vs review-weighted baseline)
    tend = []
    for name, col in PCT_METRICS.items():
        v = liked[col].dropna()
        if len(v) < MIN_N_TENDENCY:
            continue
        b = base["pct_metrics"][col]
        z = (v.mean() - b["mean"]) / (b["sd"] / np.sqrt(len(v)))
        if abs(z) >= Z_TENDENCY:
            tend.append({"metric": name, "direction": "higher" if z > 0 else "lower", "user_mean_percentile": round(float(v.mean()), 3),
                         "baseline_mean_percentile": round(b["mean"], 3), "z": round(float(z), 1), "n": int(len(v)),
                         "confidence": "high" if abs(z) >= Z_STRONG else "medium", "status": "observed_pattern"})

    # --- dietary patterns from behaviour
    constraints = {}
    for cname, col in RULES.items():
        v = liked[col].dropna()
        if len(v) < PATTERN_MIN_N:
            continue
        share, b = float(v.mean()), base["rule_rates"][col]
        if share >= PATTERN_MIN_SHARE and b < 0.9:
            z = (share - b) / np.sqrt(b * (1 - b) / len(v))
            if z >= PATTERN_Z:
                constraints[cname] = {"constraint": cname, "status": "observed_pattern", "confidence": "medium",
                                      "evidence": f"{share:.0%} of {len(v)} liked recipes satisfy '{cname}' vs {b:.0%} for a typical reviewed recipe (z={z:.1f}). Pattern only - not a confirmed restriction."}
    for c, implied in IMPLIES.items():
        if c in constraints:
            for k in implied:
                constraints.pop(k, None)

    # --- text evidence (self-reports are weak: reviews also talk about friends/family/recipes)
    mentions = {k: [] for k in MENTION_LABELS}
    health_hits, household = [], Counter()
    for _, row in rows[rows["mentions_restriction"] | rows["mentions_health"] | rows["mentions_household"]].iterrows():
        t = str(row["text"])
        for lab, pat in MENTION_LABELS.items():
            m = re.search(pat, t, re.I)
            if m:
                mentions[lab].append(snippet(t, m))
        m = HEALTH_RE.search(t)
        if m:
            health_hits.append(snippet(t, m))
        for m in HOUSEHOLD_RE.finditer(t):
            household[m.group(0).lower()] += 1
    for lab, snips in mentions.items():
        if lab == "allergy (unspecified)" and not snips:
            continue
        if len(snips) >= 2:
            ex = constraints.get(lab)
            if ex:
                ex["confidence"] = "high"
                ex["evidence"] += f" Also mentioned in {len(snips)} reviews."
            elif lab in RULES:
                constraints[lab] = {"constraint": lab, "status": "possible_self_report", "confidence": "low",
                                    "evidence": f"Mentioned in {len(snips)} reviews (may refer to other people or the recipe): " + " | ".join(snips[:2])}
            else:
                ev.append({"type": "repeated_mention", "claim": lab, "confidence": "low", "detail": snips[:2]})
        elif len(snips) == 1:
            ev.append({"type": "single_mention", "claim": lab, "confidence": "very_low", "detail": snips})
    ctx = [{"term": t, "n_reviews": int(n), "status": "possible_self_report"} for t, n in household.most_common(3) if n >= 3]

    # --- health goal: unknown unless several health mentions AND a matching 'lighter' tendency
    goal = {"value": "unknown", "status": "unknown", "confidence": None, "evidence": "No clear evidence of a health goal."}
    lighter = [t for t in tend if t["metric"] in ("calories", "sodium", "sugar", "saturated fat") and t["direction"] == "lower"]
    if len(health_hits) >= 3 and lighter:
        goal = {"value": "possibly prefers lighter eating (" + ", ".join(t["metric"] for t in lighter) + ")", "status": "observed_pattern",
                "confidence": "low", "evidence": f"{len(health_hits)} reviews mention health topics and liked recipes skew lower in those nutrients. Not confirmed."}
    else:
        unknown.append("health_goal")
        if health_hits:
            ev.append({"type": "health_mentions", "claim": f"{len(health_hits)} reviews mention health topics", "confidence": "very_low", "detail": health_hits[:2]})

    dur = liked["duration"].dropna()
    time_budget = {"value": None, "status": "unknown", "typical_minutes": float(dur.median()) if len(dur) >= MIN_N_TENDENCY else None,
                   "evidence": "median duration of liked recipes - typical, not a stated limit"}
    unknown.append("time_budget")

    p = {
        "profile_id": f"H{i:03d}", "source": "historical", "label": label,
        "dietary_constraints": list(constraints.values()), "health_goal": goal, "time_budget_minutes": time_budget,
        "taste": {"likes": likes, "dislikes": dislikes}, "nutrition_tendencies": tend, "household_context": ctx,
        "unknown_fields": unknown, "tension": None,
        "history": {"source_user_id": int(uid), "n_reviews": int(len(rows)), "n_liked": int(len(liked)), "n_disliked": int(len(disliked)),
                    "span_years": float(ustats["span_years"]), "mean_rating": float(rows["rating"].mean()), "rating_std": float(rows["rating"].std()),
                    "feature_coverage": round(coverage, 3), "n_modification_mentions": int(rows["mentions_modification"].sum())},
        "evidence": ev,
    }
    p["profile_text"] = render_profile_text(p)
    return p

In [7]:

base = json.load(open(PROC_DIR / "baselines.json"))
us = read_table("user_profile_candidates")
sel = select_users(us)
ids = set(sel["new_member_id"])
rv = read_table("reviews_clean", usecols=["new_member_id", "new_recipe_id", "rating", "rating_type", "liked", "text", "has_text", "date",
                                              "mentions_restriction", "mentions_health", "mentions_household", "mentions_modification"],
                    filter_col="new_member_id", filter_values=ids)
for c in ("liked", "has_text", "mentions_restriction", "mentions_health", "mentions_household", "mentions_modification"):
        rv[c] = rv[c].astype(bool)
lookup = read_table("recipe_lookup", filter_col="new_recipe_id", filter_values=set(rv["new_recipe_id"]))
historical_profiles = []
for k, (_, row) in enumerate(sel.iterrows(), 1):
    historical_profiles.append(build_profile(k, row["new_member_id"], f"{row['richness']}/{row['signal']}",
                                      rv[rv["new_member_id"] == row["new_member_id"]], lookup, base, row))
write_jsonl(historical_profiles, "profiles_historical")
summ = pd.DataFrame([{
        "profile_id": p["profile_id"], "label": p["label"], "n_reviews": p["history"]["n_reviews"], "n_liked": p["history"]["n_liked"],
        "coverage": p["history"]["feature_coverage"], "likes": ", ".join(x["item"] for x in p["taste"]["likes"][:5]),
        "tendencies": "; ".join(f"{t['direction']} {t['metric']}" for t in p["nutrition_tendencies"]),
        "constraints": "; ".join(f"{c['constraint']}[{c['status']}]" for c in p["dietary_constraints"]),
        "goal": p["health_goal"]["value"]} for p in historical_profiles])
summ.to_csv(PROC_DIR / "profiles_historical_summary.csv", index=False)
print(summ.to_string(index=False))
print("\nSample profile text:\n", historical_profiles[0]["profile_text"])

eligible users per stratum:
 richness  signal     
rich      no_signal        75
          with_signal     691
sparse    no_signal      1829
          with_signal     873
allocation per stratum: {('rich', 'no_signal'): 8, ('rich', 'with_signal'): 8, ('sparse', 'no_signal'): 7, ('sparse', 'with_signal'): 7}
saved processed\profiles_historical.jsonl  (30 profiles)
profile_id              label  n_reviews  n_liked  coverage                                                                                         likes                                                                                         tendencies                                                        constraints                                                              goal
      H001     rich/no_signal        173      164     0.942              hand-formed-cookies, cookies-and-brownies, desserts, icing sugar, almond extract                                                                  higher saturated fat; lower fi

### SYNTHETIC profiles (30, explicitly specified, systematically varied) on the same schema as historical ones.

30 hand-designed archetypes: 14 standard, 8 deliberate CONFLICTS between priorities, 6 MISSING-information profiles and
2 explicit-NONE profiles ('none' is a stated fact, 'unknown' is not). Optional --n-extra adds random draws from the factor grid.
Each profile is checked against the catalog for feasibility (enough recipes that satisfy its hard constraints).

* Inputs : processed/catalog

* Outputs: processed/profiles_synthetic.jsonl, profiles_synthetic_summary.csv

In [8]:
MIN_FEASIBLE = 30

# term label -> regex over normalized ingredients / informative tags (stored in the profile so checks are reproducible)
TERMS = {
    "pasta": r"pasta|spaghetti|macaroni|penne|noodle", "chicken": r"chicken", "beans": r"\bbeans?\b|lentil|chickpea",
    "cheese": r"cheese|parmesan|mozzarella|cheddar", "rice": r"\brice\b", "spicy": r"chili|chile|jalapeno|cayenne|curry|sriracha",
    "salad": r"salad|lettuce|arugula", "soup": r"\bsoup|stew|broth", "chocolate": r"chocolate|cocoa", "potato": r"potato",
    "tofu": r"tofu|tempeh", "salmon": r"salmon|\bfish\b", "creamy": r"cream|cheese|butter", "mushroom": r"mushroom",
    "cilantro": r"cilantro|coriander", "olives": r"olive(?!\s*oil)", "onion": r"onion", "seafood": r"shrimp|crab|salmon|\bfish\b|tuna",
    "soy sauce": r"soy sauce", "cured meat": r"bacon|ham\b|sausage|salami", "baking": r"flour|baking", "tomato": r"tomato",
}

In [9]:

def C(*names):
    return [{"constraint": n, "status": "stated", "confidence": "stated", "evidence": "specified by design"} for n in names]

In [10]:

ARCHETYPES = [
    # (name, group, constraints, goal, time_limit, likes, dislikes, tension)
    # constraints: None = unknown, [] = explicitly none | goal: None = unknown, "none" = explicitly none
    # time_limit : None = unknown, "none" = explicitly no limit, int = minutes
    # ---- standard (14): clear, non-conflicting specs
    ("busy_vegetarian", "standard", ["vegetarian"], None, 30, ["pasta", "cheese"], ["mushroom"], None),
    ("sodium_watcher", "standard", [], "lower sodium", 45, ["chicken", "salad"], [], None),
    ("vegan_protein", "standard", ["vegan"], "higher protein", 45, ["beans", "tofu"], [], None),
    ("nut_free_family", "standard", ["nut-free"], None, 30, ["pasta", "chicken"], ["seafood"], None),
    ("calorie_conscious_quick", "standard", [], "fewer calories", 30, ["salad", "soup"], ["cheese"], None),
    ("gluten_free_baker", "standard", ["gluten-free"], None, 60, ["chocolate", "potato"], [], None),
    ("dairy_free_low_sugar", "standard", ["dairy-free"], "less sugar", 45, ["rice", "chicken"], [], None),
    ("vegetarian_fiber", "standard", ["vegetarian"], "more fiber", 45, ["beans", "soup"], ["mushroom"], None),
    ("low_satfat_chicken", "standard", [], "less saturated fat", 60, ["chicken", "rice"], ["cured meat"], None),
    ("vegan_dessert_lover", "standard", ["vegan"], "none", 60, ["chocolate", "baking"], [], None),
    ("gluten_free_protein", "standard", ["gluten-free"], "higher protein", 45, ["chicken", "salmon"], [], None),
    ("nut_free_quick_lunch", "standard", ["nut-free"], "none", 15, ["salad", "tomato"], [], None),
    ("vegetarian_nut_free", "standard", ["vegetarian", "nut-free"], "none", 45, ["pasta", "tomato"], [], None),
    ("dairy_free_vegetarian_fiber", "standard", ["dairy-free", "vegetarian"], "more fiber", 60, ["rice", "beans"], [], None),
    # ---- conflict (8): stated priorities pull against each other
    ("comfort_vs_light", "conflict", [], "fewer calories", 45, ["creamy", "pasta", "cheese"], [], "stated taste (rich, creamy food) pulls against stated goal (fewer calories)"),
    ("quick_vegan_protein", "conflict", ["vegan"], "higher protein", 15, ["beans"], [], "tight time limit + vegan + protein goal leaves few options"),
    ("low_sodium_loves_salty", "conflict", [], "lower sodium", 45, ["soy sauce", "cured meat"], [], "likes salty foods but wants lower sodium"),
    ("sweet_tooth_less_sugar", "conflict", [], "less sugar", 60, ["chocolate", "baking"], [], "loves sweet baking but wants less sugar"),
    ("low_satfat_loves_cheese", "conflict", [], "less saturated fat", 45, ["cheese", "creamy"], [], "likes cheese/cream but wants less saturated fat"),
    ("fewer_calories_loves_carbs", "conflict", [], "fewer calories", 60, ["pasta", "potato"], [], "likes starchy comfort food but wants fewer calories"),
    ("quick_gluten_free_protein", "conflict", ["gluten-free"], "higher protein", 15, ["chicken"], [], "very short time + gluten-free + protein goal"),
    ("quick_vegetarian_protein", "conflict", ["vegetarian"], "higher protein", 15, ["beans", "cheese"], [], "short time and a protein goal while meat-free"),
    # ---- missing information (6): unknown fields are explicit
    ("minimal_info", "missing_info", None, None, 30, [], [], None),
    ("taste_only_spicy", "missing_info", None, None, None, ["spicy", "rice"], [], None),
    ("goal_only_sodium", "missing_info", None, "lower sodium", None, [], [], None),
    ("constraint_only_vegan", "missing_info", ["vegan"], None, None, [], [], None),
    ("time_and_soup", "missing_info", None, None, 45, ["soup"], [], None),
    ("dislikes_only", "missing_info", None, None, None, [], ["mushroom", "onion"], None),
    # ---- explicit none (2): 'none' is a stated fact, not 'unknown'
    ("no_restrictions_baseline", "explicit_none", [], "none", "none", ["pasta", "tomato"], [], None),
    ("easygoing_no_constraints", "explicit_none", [], "none", "none", ["rice"], ["onion"], None),
]
assert len(ARCHETYPES) == 30

In [11]:
def make_profile(pid, name, label, cons, goal, time_limit, likes, dislikes, tension, known_constraints=True):
    unknown = []
    if cons is None:
        dc, cons = [], []
        unknown += ["dietary_constraints", "allergies"]
    else:
        dc = C(*cons)
    if goal is None:
        hg = {"value": "unknown", "status": "unknown", "confidence": None, "evidence": "not provided"}
        unknown.append("health_goal")
    elif goal == "none":
        hg = {"value": "none", "status": "stated", "confidence": "stated", "evidence": "specified by design"}
    else:
        col, op, thr = GOAL_RULES[goal]
        hg = {"value": goal, "status": "stated", "confidence": "stated", "evidence": "specified by design",
              "operational_definition": f"{col} {op} {thr}"}
    if time_limit == "none":
        tb = {"value": None, "status": "stated", "typical_minutes": None, "evidence": "no time limit, specified by design"}
    elif time_limit:
        tb = {"value": time_limit, "status": "stated", "typical_minutes": None, "evidence": "specified by design"}
    else:
        tb = {"value": None, "status": "unknown", "typical_minutes": None, "evidence": "not provided"}
        unknown.append("time_budget")
    if not likes and not dislikes:
        unknown.append("taste_preferences")
    mk = lambda items: [{"item": i, "kind": "term", "match_regex": TERMS[i], "status": "stated"} for i in items]
    p = {"profile_id": pid, "source": "synthetic", "label": f"{label}/{name}", "dietary_constraints": dc, "health_goal": hg,
         "time_budget_minutes": tb, "taste": {"likes": mk(likes), "dislikes": mk(dislikes)}, "nutrition_tendencies": [],
         "household_context": [], "unknown_fields": unknown, "tension": tension, "history": None,
         "evidence": [{"type": "specified_by_design", "claim": "all attributes explicitly set", "confidence": "stated"}]}
    return p


In [12]:
def feasibility(cat, p):
    cons = [c["constraint"] for c in p["dietary_constraints"]]
    t = p["time_budget_minutes"]["value"]
    hard = constraint_mask(cat, cons, t)
    out = {"n_hard_feasible": int(hard.sum()), "n_verified": int((verified_mask(cat, cons) & hard).sum())}
    like = pd.Series(False, index=cat.index)
    for x in p["taste"]["likes"]:
        like |= term_mask(cat, x["match_regex"])
    dis = pd.Series(False, index=cat.index)
    for x in p["taste"]["dislikes"]:
        dis |= term_mask(cat, x["match_regex"])
    out["n_hard_and_liked"] = int((hard & like & ~dis).sum()) if p["taste"]["likes"] else None
    g = p["health_goal"]["value"]
    out["n_hard_and_goal"] = int((hard & goal_mask(cat, g)).sum()) if g in GOAL_RULES else None
    out["n_all_three"] = int((hard & like & ~dis & goal_mask(cat, g)).sum()) if (g in GOAL_RULES and p["taste"]["likes"]) else None
    return out

In [13]:
class Args: n_extra = 0 # Change it to whatever number of extra profiles you want
args = Args()

rng = np.random.default_rng(SEED)
cat = read_table("catalog")
parse_list_cols(cat)
cat["tags_str"] = cat["tags_str"].fillna("")
cat["ing_text"] = cat["ing_text"].fillna("")

profiles_synthetic = [make_profile(f"S{i:03d}", *a) for i, a in enumerate(ARCHETYPES, 1)]
cons_grid = [[], ["vegetarian"], ["vegan"], ["dairy-free"], ["gluten-free"], ["nut-free"], ["vegetarian", "nut-free"]]
goals = list(GOAL_RULES) + ["none", None]
times = [None, 15, 30, 45, 60]
like_pool = ["pasta", "chicken", "beans", "cheese", "rice", "spicy", "salad", "soup", "chocolate", "potato", "tofu", "salmon"]
dis_pool = ["mushroom", "cilantro", "olives", "onion", "tomato", "seafood"]
n_done, tries = 0, 0
while n_done < args.n_extra and tries < 500:
        tries += 1
        cons = cons_grid[rng.integers(len(cons_grid))]
        goal = goals[rng.integers(len(goals))]
        t = times[rng.integers(len(times))]
        lk = list(rng.choice(like_pool, rng.integers(0, 3), replace=False))
        dk = list(rng.choice(dis_pool, rng.integers(0, 2), replace=False))
        p = make_profile(f"S{len(profiles_synthetic) + 1:03d}", f"grid{n_done + 1}", "random_grid", cons, goal, t, lk, dk, None)
        if feasibility(cat, p)["n_hard_feasible"] >= MIN_FEASIBLE:
            profiles_synthetic.append(p)
            n_done += 1

rows = []
for p in profiles_synthetic:
    f = feasibility(cat, p)
    p["feasibility"] = f
    p["profile_text"] = render_profile_text(p)
    rows.append({"profile_id": p["profile_id"], "label": p["label"], **f})
    if f["n_hard_feasible"] < MIN_FEASIBLE:
        print(f"WARNING {p['profile_id']} {p['label']}: only {f['n_hard_feasible']} recipes satisfy the hard constraints - loosen it or enlarge the catalog")
for p in profiles_synthetic:
    f = p["feasibility"]
    for k in ("n_hard_and_liked", "n_hard_and_goal", "n_all_three"):
        if f[k] is not None and f[k] < 10:
            print(f"WARNING {p['profile_id']} {p['label']}: {k}={f[k]} (<10) - recommender will struggle to find good matches; revisit the likes/goal terms")
write_jsonl(profiles_synthetic, "profiles_synthetic")
summ = pd.DataFrame(rows)
summ.to_csv(PROC_DIR / "profiles_synthetic_summary.csv", index=False)
print(summ.to_string(index=False))
print("\nSample profile text:\n", profiles_synthetic[14]["profile_text"])

WARNING S016 conflict/quick_vegan_protein: n_all_three=2 (<10) - recommender will struggle to find good matches; revisit the likes/goal terms
saved processed\profiles_synthetic.jsonl  (30 profiles)
profile_id                                  label  n_hard_feasible  n_verified  n_hard_and_liked  n_hard_and_goal  n_all_three
      S001               standard/busy_vegetarian             1747         450             421.0              NaN          NaN
      S002                standard/sodium_watcher             3812        3812             941.0           1589.0        235.0
      S003                 standard/vegan_protein              789         178             126.0             50.0         13.0
      S004               standard/nut_free_family             2219        2219             418.0              NaN          NaN
      S005       standard/calorie_conscious_quick             2550        2550             374.0           1236.0        183.0
      S006             standard/gluten_f